# Value labeling (parallel)

Faster version: API calls run **concurrently** across a thread pool (`MAX_WORKERS`). Same inputs, outputs, and resume behavior as `4_value_extraction.ipynb` - only the calling is parallelized.

Assigns values from a fixed taxonomy to each story moral, using one or more
models (default **GPT + Gemini**). Reads `tidy/morals_long.csv` (one row per
moral) and writes the **same table plus one `Values` column** - a JSON object
mapping each labeling model to its chosen taxonomy labels, e.g.

`{"gpt-5.4": ["Harm", "Power"], "gemini-3.1-pro-preview": ["Power", "Authority"]}`

It stays one moral per row (NOT exploded to one value per row).

- **Input:**  `tidy/morals_long.csv`  (from 3_parse_outputs)
- **Taxonomy:** `Values_Taxonomy.csv`  (column 1 = the label set)
- **Output:** `moral_values.csv`  = morals_long columns + `Values`

Run after 3_parse_outputs. Checkpoints per moral and resumes per (moral, model).
API keys are live `getpass` prompts.

## 1. Imports + parameters

In [1]:
import re, json, time, random
from hashlib import sha256
from getpass import getpass
from pathlib import Path

import pandas as pd
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
# --- Paths -------------------------------------------------------------------
MORALS_LONG  = Path("tidy/morals_long.csv")   # input (one row per moral)
TAXONOMY_CSV = Path("Values_Taxonomy.csv")    # column 1 holds the labels
OUTPUT_CSV   = Path("moral_values.csv")       # morals_long + `Values`

# --- Labeling models ---------------------------------------------------------
# Each moral is labeled by every model below; `Values` is a JSON object keyed
# by model. Comment out a line to drop that model (provider inferred from name).
LABELING_MODELS = [
    "gpt-5.4",
    "gemini-3.1-pro-preview",
]

# --- Generation settings -----------------------------------------------------
TEMPERATURE       = 0
USE_TEMPERATURE   = True     # set False for reasoning models that reject `temperature`
MAX_OUTPUT_TOKENS = 512

# Shuffle the taxonomy label order per moral (reduces list-order bias). The
# shuffle is deterministic (seeded from RUN_SEED + the moral's identity) and is
# shared across models for a given moral, so all models see the same order.
SHUFFLE_LABELS = True
RUN_SEED       = 20240601

# --- Robustness / rate limiting ----------------------------------------------
DELAY_SECONDS   = 0.5
MAX_RETRIES     = 5
REQUEST_TIMEOUT = 120
MAX_WORKERS     = 8          # concurrent API calls -- the speed-up lever
SAVE_EVERY      = 10         # checkpoint after every N completed calls

## 2. Taxonomy + prompt

In [3]:
# Label set = first column of the taxonomy, verbatim.
LABEL_LIST = pd.read_csv(TAXONOMY_CSV).iloc[:, 0].dropna().astype(str).tolist()
print(f"{len(LABEL_LIST)} taxonomy labels loaded.")


def build_label_prompt(moral_text, labels):
    label_block = "\n- ".join(labels)
    return (
        "You are annotating the moral of a story using a fixed taxonomy of values.\n\n"
        "Select ALL labels from the taxonomy that apply. Multiple labels are allowed. "
        "Use ONLY labels from this list (verbatim):\n\n"
        f"- {label_block}\n\n"
        f'Moral to annotate:\n"""\n{moral_text}\n"""\n\n'
        'Return ONLY a JSON array of the chosen label strings, e.g. ["Label A", "Label B"]. '
        "No prose, no explanation, no markdown."
    )


def row_seed(generation_id, moral_index):
    h = sha256(f"{RUN_SEED}|{generation_id}|{moral_index}".encode())
    return int.from_bytes(h.digest()[:4], "big")


def shuffled_labels(labels, seed):
    lab = list(labels)
    random.Random(seed).shuffle(lab)
    return lab

62 taxonomy labels loaded.


## 3. API keys + model calls

Prompts for whichever keys the configured models need (OpenAI and/or Gemini).

In [4]:
def provider_for_model(model):
    m = model.lower()
    if m.startswith(("gpt", "o1", "o3")): return "openai"
    if m.startswith("gemini"):            return "google"
    raise ValueError(f"Unsupported model: {model}")

_providers = {provider_for_model(m) for m in LABELING_MODELS}
openai_client = None
gemini_key = None
if "openai" in _providers:
    openai_client = OpenAI(api_key=getpass("Enter your OpenAI API key: "))
if "google" in _providers:
    gemini_key = getpass("Enter your Gemini (Google) API key: ")


def _call_openai(model, prompt):
    kwargs = dict(model=model, messages=[{"role": "user", "content": prompt}],
                  max_completion_tokens=MAX_OUTPUT_TOKENS, timeout=REQUEST_TIMEOUT)
    if USE_TEMPERATURE:
        kwargs["temperature"] = TEMPERATURE
    return openai_client.chat.completions.create(**kwargs).choices[0].message.content


def _call_gemini(model, prompt):
    import requests
    gen = {}  # no JSON-mode: the prompt asks for a bare JSON array (parsed below)
    if USE_TEMPERATURE:
        gen["temperature"] = TEMPERATURE
    r = requests.post(
        f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent",
        headers={"x-goog-api-key": gemini_key, "content-type": "application/json"},
        json={"contents": [{"parts": [{"text": prompt}]}], "generationConfig": gen},
        timeout=REQUEST_TIMEOUT)
    if r.status_code != 200:
        raise RuntimeError(f"Gemini {r.status_code}: {r.text[:300]}")
    data = r.json()
    cands = data.get("candidates") or []
    if not cands:
        raise RuntimeError(f"Gemini: no candidates ({json.dumps(data)[:300]})")
    cand = cands[0]
    parts = (cand.get("content") or {}).get("parts") or []
    # Join all real (non-thought) text parts; reasoning models may emit several.
    texts = [p["text"] for p in parts
             if isinstance(p, dict) and "text" in p and not p.get("thought")]
    if not texts:
        raise RuntimeError("Gemini: empty reply "
                           f"(finishReason={cand.get('finishReason')}; raw={json.dumps(data)[:300]})")
    return "".join(texts)


def call_model(model, prompt):
    provider = provider_for_model(model)
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            if provider == "openai": return _call_openai(model, prompt)
            if provider == "google": return _call_gemini(model, prompt)
        except Exception as e:
            transient = any(s in str(e).lower() for s in
                            ("429", "500", "502", "503", "529", "timeout", "overloaded"))
            if attempt == MAX_RETRIES or not transient:
                raise
            wait = 2 ** attempt
            print(f"      {model} error ({e}); retry in {wait}s ({attempt}/{MAX_RETRIES})")
            time.sleep(wait)


def extract_json_array(txt):
    # Strip code fences, keep the LAST [...] block, parse to a list of strings.
    if not txt:
        return None
    t = re.sub(r"```(json)?", "", txt)
    arrays = re.findall(r"\[.*?\]", t, flags=re.S)
    if not arrays:
        return None
    try:
        parsed = json.loads(arrays[-1])
    except Exception:
        return None
    return [str(x) for x in parsed]

Enter your OpenAI API key:  ········
Enter your Gemini (Google) API key:  ········


## 4. Label every moral with every model (checkpoint / resume)

`Values` = a JSON object `{model: [labels]}`. Reruns reuse any (moral, model)
already present, so adding a model only fills the missing one.

In [5]:
df = pd.read_csv(MORALS_LONG)

# Resume: parse existing Values objects so we can reuse per-(moral, model).
existing = {}
if OUTPUT_CSV.exists():
    prev = pd.read_csv(OUTPUT_CSV)
    if "Values" in prev.columns:
        for r in prev.itertuples():
            key = (str(r.generation_id), int(r.moral_index))
            try:
                existing[key] = json.loads(r.Values) if isinstance(r.Values, str) and r.Values.strip() else {}
            except Exception:
                existing[key] = {}

# Stable row order + per-moral result store (seeded from anything already done).
keys = [(str(r.generation_id), int(r.moral_index)) for r in df.itertuples()]
results = {k: dict(existing.get(k, {})) for k in keys}

# Build the list of (moral, model) calls still needed (non-empty result = done).
tasks = []
for r in df.itertuples():
    key = (str(r.generation_id), int(r.moral_index))
    labels = shuffled_labels(LABEL_LIST, row_seed(*key)) if SHUFFLE_LABELS else LABEL_LIST
    for model in LABELING_MODELS:
        val = results[key].get(model)
        if isinstance(val, list) and len(val) > 0:   # already labeled (non-empty)
            continue
        tasks.append((key, model, r.moral_text, labels))

print(f"{len(tasks)} call(s) to make across {MAX_WORKERS} workers "
      f"({len(df)} morals x {len(LABELING_MODELS)} models, minus already done).")


def _label_one(task):
    key, model, moral_text, labels = task
    try:
        raw = call_model(model, build_label_prompt(moral_text, labels))
        return key, model, (extract_json_array(raw) or []), None
    except Exception as e:
        return key, model, None, str(e)


def _write():
    out = df.copy()
    out["Values"] = [json.dumps(results[k], ensure_ascii=False) for k in keys]
    out.to_csv(OUTPUT_CSV, index=False)


# Run the calls concurrently; collect as they finish; checkpoint periodically.
done_n = 0
if tasks:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = [ex.submit(_label_one, t) for t in tasks]
        for fut in as_completed(futures):
            key, model, vals, err = fut.result()
            results[key][model] = vals
            done_n += 1
            print(f"[{done_n}/{len(tasks)}] {key} / {model} -> " + ("OK" if err is None else f"ERROR: {err}"))
            if done_n % SAVE_EVERY == 0:
                _write()

_write()
print(f"\nDone. {len(df)} moral(s) labeled by {LABELING_MODELS} -> {OUTPUT_CSV}")

36 call(s) to make across 8 workers (18 morals x 2 models, minus already done).
[1/36] ('Test__full_text__gpt-5.4', 1) / gpt-5.4 -> OK
[2/36] ('Test__full_text__gpt-5.4', 2) / gpt-5.4 -> OK
[3/36] ('Test__full_text__gpt-5.4', 3) / gpt-5.4 -> OK
[4/36] ('Test__full_text__gemini-3.1-pro-preview', 1) / gpt-5.4 -> OK
[5/36] ('Test__full_text__gemini-3.1-pro-preview', 2) / gpt-5.4 -> OK
[6/36] ('Test__full_text__gemini-3.1-pro-preview', 3) / gpt-5.4 -> OK
[7/36] ('Test__chunk_summary__gpt-5.4', 1) / gpt-5.4 -> OK
[8/36] ('Test__chunk_summary__gpt-5.4', 2) / gpt-5.4 -> OK
[9/36] ('Test__full_text__gpt-5.4', 1) / gemini-3.1-pro-preview -> OK
[10/36] ('Test__chunk_summary__gpt-5.4', 3) / gpt-5.4 -> OK
[11/36] ('Test__full_text__gpt-5.4', 2) / gemini-3.1-pro-preview -> OK
[12/36] ('Test__chunk_summary__gemini-3.1-pro-preview', 1) / gpt-5.4 -> OK
[13/36] ('Test__full_text__gemini-3.1-pro-preview', 2) / gemini-3.1-pro-preview -> OK
[14/36] ('Test__chunk_summary__gemini-3.1-pro-preview', 2) / gpt-

## 5. Inspect

In [ ]:
pd.read_csv(OUTPUT_CSV).head(12)